# Day 1 · Play Store Raw Dataset Ingestion & Schema Profiling
**Project:** Unveiling the Android App Market (Google Play Store Analysis)  
**Task Checklist Coverage:**
- [x] Load the Play Store apps dataset and the user reviews dataset separately
- [x] Structural schema inspection (`.info()`, `.head()`, `.describe()`)
- [x] Detect corrupted records (identifying shifted row 10472 where Category='1.9')
- [x] Profile data types and identify non-numeric strings in numeric columns (`Installs`, `Price`, `Reviews`, `Size`)
- [x] Profile missing values across both datasets
- [x] Inspect user review sentiment baseline


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Data directory
DATA_DIR = Path('../Data')
apps_csv = DATA_DIR / 'googleplaystore.csv'
reviews_csv = DATA_DIR / 'googleplaystore_user_reviews.csv'

print(f"Apps path exists: {apps_csv.exists()}")
print(f"Reviews path exists: {reviews_csv.exists()}")

Apps path exists: True
Reviews path exists: True


## 1. Separate Dataset Ingestion
Loading the Google Play Store apps catalog and customer sentiment reviews separately.

In [2]:
df_apps = pd.read_csv(apps_csv)
df_reviews = pd.read_csv(reviews_csv)

print(f"Apps Dataset Dimensions: {df_apps.shape[0]:,} rows × {df_apps.shape[1]} columns")
print(f"User Reviews Dimensions: {df_reviews.shape[0]:,} rows × {df_reviews.shape[1]} columns")

Apps Dataset Dimensions: 10,841 rows × 13 columns
User Reviews Dimensions: 64,295 rows × 5 columns


## 2. Apps Dataset Schema & Column Profiling

In [3]:
df_apps.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 2.2 MB


In [4]:
df_apps.head(5)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## 3. Detecting Corrupted / Shifted Rows (Row 10472)
A known issue in raw Play Store datasets is a corrupted row where missing category information caused a leftward column shift.

In [5]:
# Detect any rows where Category is numeric
corrupt_rows = df_apps[df_apps['Category'].str.match(r'^[0-9]', na=False)]
print(f"Number of detected corrupted shifted rows: {len(corrupt_rows)}")
corrupt_rows

Number of detected corrupted shifted rows: 1


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


## 4. Identifying Non-Numeric Data Encodings Requiring Pipeline Transformation
Profiling columns where numerical variables are improperly encoded as strings (`Reviews`, `Installs`, `Price`, `Size`).

In [6]:
print("Sample 'Installs' values:", df_apps['Installs'].dropna().unique()[:8])
print("Sample 'Price' values:", df_apps['Price'].dropna().unique()[:8])
print("Sample 'Size' values:", df_apps['Size'].dropna().unique()[:8])

Sample 'Installs' values: <ArrowStringArray>
[    '10,000+',    '500,000+',  '5,000,000+', '50,000,000+',    '100,000+',
     '50,000+',  '1,000,000+', '10,000,000+']
Length: 8, dtype: str
Sample 'Price' values: <ArrowStringArray>
['0', '$4.99', '$3.99', '$6.99', '$1.49', '$2.99', '$7.99', '$5.99']
Length: 8, dtype: str
Sample 'Size' values: <ArrowStringArray>
['19M', '14M', '8.7M', '25M', '2.8M', '5.6M', '29M', '33M']
Length: 8, dtype: str


## 5. Missing Value Profiling

In [7]:
missing_apps = df_apps.isnull().sum()
missing_apps_pct = (missing_apps / len(df_apps)) * 100
missing_summary = pd.DataFrame({'Missing_Count': missing_apps, 'Missing_Pct': missing_apps_pct})
missing_summary[missing_summary['Missing_Count'] > 0]

,Missing_Count,Missing_Pct
Rating,1474,13.596532
Type,1,0.009224
Content Rating,1,0.009224
Current Ver,8,0.073794
Android Ver,3,0.027673


## 6. User Reviews Dataset Inspection & Baseline Sentiment

In [8]:
df_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 64295 entries, 0 to 64294
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   App                     64295 non-null  str    
 1   Translated_Review       37427 non-null  str    
 2   Sentiment               37432 non-null  str    
 3   Sentiment_Polarity      37432 non-null  float64
 4   Sentiment_Subjectivity  37432 non-null  float64
dtypes: float64(2), str(3)
memory usage: 8.3 MB


In [9]:
df_reviews.head(5)

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [10]:
print("Review Sentiment Distribution (Raw):")
print(df_reviews['Sentiment'].value_counts(dropna=False))
print(f"\nNull Translated_Review Count: {df_reviews['Translated_Review'].isnull().sum():,} ({df_reviews['Translated_Review'].isnull().mean()*100:.2f}%)")

Review Sentiment Distribution (Raw):
Sentiment
NaN         26863
Positive    23998
Negative     8271
Neutral      5163
Name: count, dtype: int64

Null Translated_Review Count: 26,868 (41.79%)


## Summary of Day 1 Findings
1. **Catalog Dimensions:** 10,841 raw app entries across 13 columns; 64,295 customer review records.
2. **Critical Anomaly:** Identified corrupted shifted record at index 10472 (`Category` = '1.9', `Reviews` = '3.0M') requiring column realignment or removal.
3. **Type Normalization Target:** `Installs` has `+` and `,` chars; `Price` has `$` symbols; `Size` has `M`, `k`, and `Varies with device`.
4. **Missing Values:** `Rating` is missing in 13.6% of records. User reviews contain ~41.7% null translations.
5. **Next Step (Day 2):** Construct the modular data cleaning pipeline to execute type conversions, resolve anomalies, and handle null values.